In [2]:
import pandas as pd 
alumni_df = pd.read_csv('alumnis_export_20260120_134911.csv', encoding='utf-8')
alumni_df.head()

,ID,Nom,Prénom,Ville,Poste,LinkedIn,Promo,Latitude,Longitude,Entreprise
0,1,Garcia,Jonathan,Portets,Artisan d’art,https://linkedin.com/in/jonathan-garcia-362b65357,2010,44.695969,-0.423824,Soleil Verre
1,2,THIERY,Marie,Strasbourg,"Chargée de mission ""Supports de Communication""",https://linkedin.com/in/marie-thiery-b4912649,2025,48.584614,7.750713,ARTE
2,3,Crochard,Maureen,Toulon,Chargée de communication - Apprentie,https://linkedin.com/in/maureen-crochard-03281...,2025,43.125731,5.930492,Université de Toulon
3,4,Garay,Laurent,Paris,Senior Project Manager 360°,https://linkedin.com/in/laurent-garay-460901141,2020,48.858890,2.320041,TBWA Paris
4,5,Pasqualini,Benjamin,Mirabeau,Chef de projet,https://linkedin.com/in/benjamin-pasqualini-81...,2019,44.058056,6.063851,PETRA PATRIMONIA


In [25]:
# Récupérer les entreprises et compter les occurrences
# Exclure "Aucune"

enterprise_df = alumni_df[alumni_df['Entreprise'] != 'Aucune']
enterprise_df = enterprise_df.groupby('Entreprise').size().reset_index(name='nombre')
enterprise_df = enterprise_df.sort_values('nombre', ascending=False).reset_index(drop=True)
enterprise_df.columns = ['nom', 'nombre']

print(f"Nombre d'entreprises uniques : {len(enterprise_df)}")
enterprise_df.head(20)

Nombre d'entreprises uniques : 721


,nom,nombre
0,Orange Business,6
1,Marine Nationale,5
2,STRATIS,4
3,Université de Toulon,4
4,L'Oréal,3
5,CGI,3
6,Freelance,3
7,COMMLAB Marine nationale,3
8,Airbus,3
9,Swello,3


## Scraping Orange Business	


In [26]:
import requests
from bs4 import BeautifulSoup

def get_sitemap_urls(sitemap_url):
    """Récupère les URLs des sitemaps enfants à partir du sitemap_index"""
    try:
        response = requests.get(sitemap_url)
        soup = BeautifulSoup(response.content, 'xml')
        urls = [loc.text for loc in soup.find_all('loc')]
        return urls
    except Exception as e:
        print(f"Erreur en récupérant {sitemap_url}: {e}")
        return []

def get_job_urls(sitemap_url):
    """Récupère les URLs des offres d'emploi à partir d'un sitemap"""
    try:
        response = requests.get(sitemap_url)
        soup = BeautifulSoup(response.content, 'xml')
        urls = [loc.text for loc in soup.find_all('loc')]
        job_urls = [u for u in urls if '/job/' in u or '/offres/' in u]
        return job_urls
    except Exception as e:
        print(f"Erreur en récupérant {sitemap_url}: {e}")
        return []

# Récupérer le sitemap_index
sitemap_index = "https://orange.jobs/fr/fr/sitemap_index.xml"
sitemap_urls = get_sitemap_urls(sitemap_index)

print(f"Nombre de sitemaps trouvés : {len(sitemap_urls)}")

# Récupérer toutes les offres d'emploi
all_job_urls = []
for sitemap_url in sitemap_urls:
    print(f"Traitement : {sitemap_url}")
    job_urls = get_job_urls(sitemap_url)
    all_job_urls.extend(job_urls)
    print(f"  -> {len(job_urls)} offres trouvées")

print(f"\nTotal d'offres d'emploi : {len(all_job_urls)}")

# Créer un DataFrame avec les URLs
orange_jobs_df = pd.DataFrame({'url': all_job_urls})

print(f"Nombre d'offres uniques : {len(orange_jobs_df)}")
print(f"\nDataFrame créé :")
orange_jobs_df.head(10)

Nombre de sitemaps trouvés : 2
Traitement : https://orange.jobs/fr/fr/sitemap1.xml
  -> 500 offres trouvées
Traitement : https://orange.jobs/fr/fr/sitemap2.xml
  -> 385 offres trouvées

Total d'offres d'emploi : 885
Nombre d'offres uniques : 885

DataFrame créé :


,url
0,https://orange.jobs/fr/fr/job/CTS-2025-47627/S...
1,https://orange.jobs/fr/fr/job/CTS-2025-45465/C...
2,https://orange.jobs/fr/fr/job/CTS-2025-43229/O...
3,https://orange.jobs/fr/fr/job/CTS-2025-46623/I...
4,https://orange.jobs/fr/fr/job/CTS-2023-26884/S...
5,https://orange.jobs/fr/fr/job/CTS-2025-47360/S...
6,https://orange.jobs/fr/fr/job/CTS-2025-47636/A...
7,https://orange.jobs/fr/fr/job/CTS-2025-47792/T...
8,https://orange.jobs/fr/fr/job/CTS-2025-46612/S...
9,https://orange.jobs/fr/fr/job/CTS-2025-47609/I...


In [27]:
import json

url = "https://orange.jobs/fr/fr/job/CTS-2025-47112/Stage-OW-OWF-Conseiller-Client-Back-Office-F-H"

def scrap_orange_job(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    
    try:
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            return f"Erreur {response.status_code}"
    except Exception as e:
        return f"Erreur de connexion : {e}"

    soup = BeautifulSoup(response.text, 'html.parser')

    # 1. Extraction du script JSON-LD
    json_script = soup.find('script', type='application/ld+json')
    
    if json_script:
        try:
            data = json.loads(json_script.string)
        except Exception as e:
            return f"Erreur parsing JSON : {e}"
        
        # 2. Préparation des données spécifiques avec gestion des listes
        try:
            job_location = data.get('jobLocation', {})
            if isinstance(job_location, list) and job_location:
                job_location = job_location[0]
            address_data = job_location.get('address', {}) if isinstance(job_location, dict) else {}
        except Exception as e:
            print(f"Erreur sur jobLocation pour {url} : {e}")
            return f"Erreur jobLocation : {e}"
        
        # Nettoyage du HTML pour le texte brut
        raw_description = data.get("description", "")
        clean_text = BeautifulSoup(raw_description, "html.parser").get_text(separator="\n").strip()

        # 3. Construction du dictionnaire final avec protection d'erreur
        try:
            job_info = {
                "Titre": data.get("title"),
                "Date de publication": data.get("datePosted"),
                "ID": data.get("identifier", {}).get("value"),
                
                # Localisation
                "Ville": address_data.get("addressLocality"),
                "Pays": address_data.get("addressCountry"),
                
                # Contenu
                "Texte": data.get("description"),
                "Lien Original": url
            }
            return job_info
        except Exception as e:
            print(f"Erreur lors de la construction du dictionnaire pour {url} : {e}")
            return f"Erreur construction dictionnaire : {e}"

    return "Données JSON non trouvées sur la page"

# --- Exécution du test ---
offre = scrap_orange_job(url)

if isinstance(offre, dict):
    print("--- RÉSULTAT DU SCRAPING ---")
    for cle, valeur in offre.items():
        if cle == "Texte":
            # On affiche seulement les 150 premiers caractères pour ne pas polluer l'écran
            print(f"{cle}: {valeur[:150]}...") 
        else:
            print(f"{cle}: {valeur}")
else:
    print(offre)

--- RÉSULTAT DU SCRAPING ---
Titre: Stage - OW/OWF Conseiller Client Back Office F/H
Date de publication: 2026-01-20
ID: CTS-2025-47112
Ville: TOULON
Pays: FRANCE
Texte: &lt;p&gt;&lt;strong&gt;Date de publication :&lt;/strong&gt; Oct 29, 2025, 3:55PM&lt;/p&gt;&lt;p&gt;Ø Participe à l'évolution du SI : Ecrit/réalise les...
Lien Original: https://orange.jobs/fr/fr/job/CTS-2025-47112/Stage-OW-OWF-Conseiller-Client-Back-Office-F-H


In [28]:
import time
import random

# Appliquer le scraping sur toutes les offres et enrichir le DataFrame
job_rows = []
errors = []
total = len(orange_jobs_df)
print(f"Début du scraping : {total} offres")

for idx, url in enumerate(orange_jobs_df['url'], start=1):
    info = scrap_orange_job(url)
    if isinstance(info, dict):
        info['url'] = url  # conserver la clé de jointure
        job_rows.append(info)
        status_msg = "OK"
    else:
        errors.append({'url': url, 'erreur': info})
        if isinstance(info, str) and '410' in info:
            status_msg = "Erreur 410"
        else:
            status_msg = f"Erreur: {info}"
            print(f"Erreur de scraping pour {url} -> {info}")
    
    print(f"Progression : {idx}/{total} | {status_msg}")

    time.sleep(random.uniform(1, 3))

orange_jobs_details = pd.DataFrame(job_rows)
orange_jobs_full = orange_jobs_df.merge(orange_jobs_details, on='url', how='left')

print(f"Offres enrichies : {len(orange_jobs_details)} / {len(orange_jobs_df)}")
print(f"Erreurs : {len(errors)}")
orange_jobs_full.head()

Début du scraping : 885 offres
Progression : 1/885 | OK
Progression : 2/885 | Erreur 410
Progression : 3/885 | OK
Progression : 4/885 | Erreur 410
Progression : 5/885 | Erreur 410
Progression : 6/885 | OK
Progression : 7/885 | Erreur 410
Progression : 8/885 | Erreur 410
Progression : 9/885 | OK
Progression : 10/885 | Erreur 410
Progression : 11/885 | Erreur 410
Progression : 12/885 | Erreur 410
Progression : 13/885 | OK
Progression : 14/885 | Erreur 410
Progression : 15/885 | Erreur 410
Progression : 16/885 | OK
Progression : 17/885 | Erreur 410
Progression : 18/885 | OK
Progression : 19/885 | Erreur 410
Progression : 20/885 | Erreur 410
Progression : 21/885 | Erreur 410
Progression : 22/885 | Erreur 410
Progression : 23/885 | Erreur 410
Progression : 24/885 | Erreur 410
Progression : 25/885 | Erreur 410
Progression : 26/885 | OK
Progression : 27/885 | Erreur 410
Progression : 28/885 | OK
Progression : 29/885 | Erreur 410
Progression : 30/885 | Erreur 410
Progression : 31/885 | Erreur 

,url,Titre,Date de publication,ID,Ville,Pays,Texte,Lien Original
0,https://orange.jobs/fr/fr/job/CTS-2025-47627/S...,Stage – IA appliquée : assistant automatique d...,2026-01-20,CTS-2025-47627,CHÂTILLON,FRANCE,&lt;p&gt;&lt;strong&gt;Date de publication :&l...,https://orange.jobs/fr/fr/job/CTS-2025-47627/S...
1,https://orange.jobs/fr/fr/job/CTS-2025-45465/C...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://orange.jobs/fr/fr/job/CTS-2025-43229/O...,Offre de stage orange Madagascar F/H,2026-01-20,CTS-2025-43229,ANTANANARIVO,MADAGASCAR,&lt;p&gt;&lt;strong&gt;Date de publication :&l...,https://orange.jobs/fr/fr/job/CTS-2025-43229/O...
3,https://orange.jobs/fr/fr/job/CTS-2025-46623/I...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://orange.jobs/fr/fr/job/CTS-2023-26884/S...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
orange_jobs_full.dropna(inplace=True)
print(f"Offres après nettoyage : {len(orange_jobs_full)}")
orange_jobs_full.drop('Lien Original', axis=1, inplace=True)
orange_jobs_full.head()

Offres après nettoyage : 542


,url,Titre,Date de publication,ID,Ville,Pays,Texte
0,https://orange.jobs/fr/fr/job/CTS-2025-47627/S...,Stage – IA appliquée : assistant automatique d...,2026-01-20,CTS-2025-47627,CHÂTILLON,FRANCE,&lt;p&gt;&lt;strong&gt;Date de publication :&l...
2,https://orange.jobs/fr/fr/job/CTS-2025-43229/O...,Offre de stage orange Madagascar F/H,2026-01-20,CTS-2025-43229,ANTANANARIVO,MADAGASCAR,&lt;p&gt;&lt;strong&gt;Date de publication :&l...
5,https://orange.jobs/fr/fr/job/CTS-2025-47360/S...,Stagiaire Ingénieur/Ingénieure - Développement...,2026-01-20,CTS-2025-47360,CAEN,FRANCE,&lt;p&gt;&lt;strong&gt;Date de publication :&l...
8,https://orange.jobs/fr/fr/job/CTS-2025-46612/S...,Stage – Ingénieur.e chercheur.euse IA agentique,2026-01-20,CTS-2025-46612,CAEN,FRANCE,&lt;p&gt;&lt;strong&gt;Date de publication :&l...
12,https://orange.jobs/fr/fr/job/CTS-2025-47055/C...,CDD - Gestionnaire Transports/logistique F/H,2026-01-20,CTS-2025-47055,BREST,FRANCE,&lt;p&gt;&lt;strong&gt;Date de publication :&l...


In [30]:
# Filtrer pour garder uniquement les offres de stage
orange_jobs_stages = orange_jobs_full[orange_jobs_full['Titre'].str.match(r'^Stage', case=False, na=False)]
print(f"Offres de stage trouvées : {len(orange_jobs_stages)} / {len(orange_jobs_full)}")
orange_jobs_stages.head(10)

Offres de stage trouvées : 34 / 542


,url,Titre,Date de publication,ID,Ville,Pays,Texte,Lien Original
0,https://orange.jobs/fr/fr/job/CTS-2025-47627/S...,Stage – IA appliquée : assistant automatique d...,2026-01-20,CTS-2025-47627,CHÂTILLON,FRANCE,&lt;p&gt;&lt;strong&gt;Date de publication :&l...,https://orange.jobs/fr/fr/job/CTS-2025-47627/S...
8,https://orange.jobs/fr/fr/job/CTS-2025-46612/S...,Stage – Ingénieur.e chercheur.euse IA agentique,2026-01-20,CTS-2025-46612,CAEN,FRANCE,&lt;p&gt;&lt;strong&gt;Date de publication :&l...,https://orange.jobs/fr/fr/job/CTS-2025-46612/S...
15,https://orange.jobs/fr/fr/job/CTS-2025-47606/S...,Stage en Communication et Valorisation de l'In...,2026-01-20,CTS-2025-47606,CHÂTILLON,FRANCE,&lt;p&gt;&lt;strong&gt;Date de publication :&l...,https://orange.jobs/fr/fr/job/CTS-2025-47606/S...
17,https://orange.jobs/fr/fr/job/CTS-2024-40152/S...,Stage – Intégration d'un Agent IA aux Outils d...,2026-01-20,CTS-2024-40152,BLAGNAC,FRANCE,&lt;p&gt;&lt;strong&gt;Date de publication :&l...,https://orange.jobs/fr/fr/job/CTS-2024-40152/S...
25,https://orange.jobs/fr/fr/job/CTS-2025-47761/S...,Stage - Graphiste F/H,2026-01-20,CTS-2025-47761,NANTERRE,FRANCE,&lt;p&gt;&lt;strong&gt;Date de publication :&l...,https://orange.jobs/fr/fr/job/CTS-2025-47761/S...
37,https://orange.jobs/fr/fr/job/CTS-2025-46917/S...,Stage Marketing - Chef de Produit Junior - Mob...,2026-01-20,CTS-2025-46917,ARCUEIL,FRANCE,&lt;p&gt;&lt;strong&gt;Date de publication :&l...,https://orange.jobs/fr/fr/job/CTS-2025-46917/S...
39,https://orange.jobs/fr/fr/job/CTS-2025-47923/S...,Stage en sociologie sur les usages de robots c...,2026-01-20,CTS-2025-47923,CHÂTILLON,FRANCE,&lt;p&gt;&lt;strong&gt;Date de publication :&l...,https://orange.jobs/fr/fr/job/CTS-2025-47923/S...
40,https://orange.jobs/fr/fr/job/CTS-2025-47287/S...,Stage - Chargé(e) du Projet de Culture Orange F/H,2026-01-20,CTS-2025-47287,ISSY-LES-MOULINEAUX,FRANCE,&lt;p&gt;&lt;strong&gt;Date de publication :&l...,https://orange.jobs/fr/fr/job/CTS-2025-47287/S...
51,https://orange.jobs/fr/fr/job/CTS-2025-47748/S...,Stage - Product Owner expérience digitale clie...,2026-01-20,CTS-2025-47748,ARCUEIL,FRANCE,&lt;p&gt;&lt;strong&gt;Date de publication :&l...,https://orange.jobs/fr/fr/job/CTS-2025-47748/S...
65,https://orange.jobs/fr/fr/job/CTS-2026-48927/S...,Stage - Chargé de Marketing opérationnel et co...,2026-01-20,CTS-2026-48927,NANTERRE,FRANCE,&lt;p&gt;&lt;strong&gt;Date de publication :&l...,https://orange.jobs/fr/fr/job/CTS-2026-48927/S...


In [ ]:
orange_jobs_stages.to_csv('offres/orange_jobs_stages.csv', index=False, encoding='utf-8')

## Scraping L'oreal


In [34]:
def get_loreal_links_only():
    base_url = "https://careers.loreal.com/fr_FR/jobs/SearchJobsAJAX/"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "X-Requested-With": "XMLHttpRequest"
    }

    job_links = []
    offset = 0
    step = 20 # Pagination par 20
    
    print("--- Phase 1 : Collecte des liens (Stages) ---")

    while True:
        # Filtre 3_33_3=135 (Stages) + Pagination
        params = {
            "3_33_3": "135",
            "jobOffset": offset
        }

        try:
            print(f"Scraping page (offset {offset})...", end="")
            response = requests.get(base_url, headers=headers, params=params)
            
            # Gestion du format de réponse (JSON contenant du HTML ou HTML direct)
            try:
                data = response.json()
                html_source = data.get('content') or data.get('html') or response.text
            except:
                html_source = response.text

            soup = BeautifulSoup(html_source, 'html.parser')
            
            # On cible les articles
            articles = soup.find_all('article', class_='article--result')
            
            if not articles:
                print(" -> Aucun résultat. Fin de la collecte.")
                break

            count_page = 0
            for article in articles:
                # On cherche le lien dans le titre h3 > a
                h3 = article.find('h3', class_='article__header__text__title')
                if h3:
                    a_tag = h3.find('a')
                    if a_tag and 'href' in a_tag.attrs:
                        link = a_tag['href']
                        job_links.append(link)
                        count_page += 1
            
            print(f" -> {count_page} liens trouvés.")
            
            # Si on trouve moins de 20 liens, c'est que c'est la dernière page
            if len(articles) < step:
                print("Dernière page atteinte.")
                break

            offset += step
            time.sleep(1) # Pause courte

        except Exception as e:
            print(f"\nErreur : {e}")
            break

    return job_links

# --- Exécution ---
loreal_job_urls = get_loreal_links_only()

# Créer un DataFrame avec les URLs
loreal_jobs_df = pd.DataFrame({'url': loreal_job_urls})

print(f"\n--- Bilan ---")
print(f"Nombre d'offres uniques trouvées : {len(loreal_jobs_df)}")
print(f"\nDataFrame créé :")
loreal_jobs_df.head(10)

--- Phase 1 : Collecte des liens (Stages) ---
Scraping page (offset 0)... -> 20 liens trouvés.
Scraping page (offset 20)... -> 20 liens trouvés.
Scraping page (offset 40)... -> 20 liens trouvés.
Scraping page (offset 60)... -> 20 liens trouvés.
Scraping page (offset 80)... -> 20 liens trouvés.
Scraping page (offset 100)... -> 20 liens trouvés.
Scraping page (offset 120)... -> 20 liens trouvés.
Scraping page (offset 140)... -> 10 liens trouvés.
Dernière page atteinte.

--- Bilan ---
Nombre d'offres uniques trouvées : 150

DataFrame créé :


,url
0,https://careers.loreal.com/fr_FR/jobs/JobDetai...
1,https://careers.loreal.com/fr_FR/jobs/JobDetai...
2,https://careers.loreal.com/fr_FR/jobs/JobDetai...
3,https://careers.loreal.com/fr_FR/jobs/JobDetai...
4,https://careers.loreal.com/fr_FR/jobs/JobDetai...
5,https://careers.loreal.com/fr_FR/jobs/JobDetai...
6,https://careers.loreal.com/fr_FR/jobs/JobDetai...
7,https://careers.loreal.com/fr_FR/jobs/JobDetai...
8,https://careers.loreal.com/fr_FR/jobs/JobDetai...
9,https://careers.loreal.com/fr_FR/jobs/JobDetai...


In [35]:
loreal_jobs_df.to_csv('loreal_jobs_stages_links.csv', index=False, encoding='utf-8')

In [37]:
import re

def get_job_date_robust(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # --- MÉTHODE 1 : JSON-LD (La pépite pour Google Jobs) ---
        json_scripts = soup.find_all('script', type='application/ld+json')
        for script in json_scripts:
            try:
                data = json.loads(script.string)
                if data.get('@type') == 'JobPosting' and 'datePosted' in data:
                    return data['datePosted']
            except:
                continue

        # --- MÉTHODE 2 : Les Meta Tags ---
        meta_date = soup.find('meta', property='article:published_time') or \
                    soup.find('meta', itemprop='datePosted') or \
                    soup.find('meta', attrs={"name": "dcterms.created"})
        
        if meta_date and meta_date.get('content'):
            return meta_date['content']

        # --- MÉTHODE 3 : Recherche de texte brut ---
        text_content = soup.get_text()
        match = re.search(r'(Publié|Posted)\s+(:|le|on)?\s*(\d{1,2}[-/\s]\w{3,10}[-/\s]\d{4})', text_content, re.IGNORECASE)
        
        if match:
            return match.group(3)

        return "Date introuvable"

    except Exception as e:
        return f"Erreur: {e}"

def scrap_loreal_details(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    try:
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            return f"Erreur {response.status_code}"

        soup = BeautifulSoup(response.text, 'html.parser')
        
        info_job = {}

        # --- ÉTAPE 1 : Extraction via le DataLayer ---
        script_content = ""
        for script in soup.find_all('script'):
            if script.string and "dataLayer.push" in script.string and "jobIDATS" in script.string:
                script_content = script.string
                break
        
        if script_content:
            def get_js_var(key, text):
                pattern = re.compile(rf'{key}:\s*"([^"]+)"')
                match = pattern.search(text)
                return match.group(1) if match else "Non trouvé"

            info_job["Titre"] = get_js_var("jobTitle", script_content)
            info_job["Ville"] = get_js_var("jobLocation", script_content)
            info_job["Pays"] = get_js_var("jobCountry", script_content)
            info_job["ID"] = get_js_var("jobIDATS", script_content)
            info_job["Type"] = get_js_var("jobPositionType", script_content)

        # --- ÉTAPE 2 : Extraction de la Date (Méthode robuste) ---
        info_job["Date de publication"] = get_job_date_robust(url)

        # --- ÉTAPE 3 : Extraction du Texte de l'offre ---
        description_div = soup.find('div', class_='article__content')
        if not description_div:
            description_div = soup.find('div', class_='job-description')
        
        if description_div:
            info_job["Texte de l'offre"] = description_div.get_text(separator='\n').strip()
        else:
            info_job["Texte de l'offre"] = "Description introuvable"

        return info_job

    except Exception as e:
        return f"Erreur critique : {e}"

# --- TEST ---
url_test = "https://careers.loreal.com/fr_FR/jobs/JobDetail/Trade-Marketing-Trainee-L-Or-al-Dermatological-Beauty-Espoo-FI/198643"
details = scrap_loreal_details(url_test)

print("--- RÉSULTATS ---")
for k, v in details.items():
    if k == "Texte de l'offre":
        print(f"{k} : {v[:200]} [...]")
    else:
        print(f"{k} : {v}")

--- RÉSULTATS ---
Titre : Trade Marketing Trainee - L'Oréal Dermatological Beauty - Espoo (FI)
Ville : Espoo, Southwest Finland
Pays : Finland
ID : 198643
Type : Stage
Date de publication : 2025-02-12
Texte de l'offre : Stage
                                                                                    







                                                                                            Southwest  [...]


In [38]:
import time
import random

# Appliquer le scraping sur toutes les offres L'oreal et enrichir le DataFrame
job_rows = []
errors = []
total = len(loreal_jobs_df)
print(f"Début du scraping L'oreal : {total} offres")

for idx, url in enumerate(loreal_jobs_df['url'], start=1):
    info = scrap_loreal_details(url)
    if isinstance(info, dict):
        info['url'] = url  # conserver la clé de jointure
        job_rows.append(info)
        status_msg = "OK"
    else:
        errors.append({'url': url, 'erreur': info})
        if isinstance(info, str) and '410' in info:
            status_msg = "Erreur 410"
        else:
            status_msg = f"Erreur: {info}"
            print(f"Erreur de scraping pour {url} -> {info}")
    
    print(f"Progression : {idx}/{total} | {status_msg}")

    time.sleep(random.uniform(1, 3))

loreal_jobs_details = pd.DataFrame(job_rows)
loreal_jobs_full = loreal_jobs_df.merge(loreal_jobs_details, on='url', how='left')

print(f"\nOffres enrichies : {len(loreal_jobs_details)} / {len(loreal_jobs_df)}")
print(f"Erreurs : {len(errors)}")
loreal_jobs_full.head()

Début du scraping L'oreal : 150 offres
Progression : 1/150 | OK
Progression : 2/150 | OK
Progression : 3/150 | OK
Progression : 4/150 | OK
Progression : 5/150 | OK
Progression : 6/150 | OK
Progression : 7/150 | OK
Progression : 8/150 | OK
Progression : 9/150 | OK
Progression : 10/150 | OK
Progression : 11/150 | OK
Progression : 12/150 | OK
Progression : 13/150 | OK
Progression : 14/150 | OK
Progression : 15/150 | OK
Progression : 16/150 | OK
Progression : 17/150 | OK
Progression : 18/150 | OK
Progression : 19/150 | OK
Progression : 20/150 | OK
Progression : 21/150 | OK
Progression : 22/150 | OK
Progression : 23/150 | OK
Progression : 24/150 | OK
Progression : 25/150 | OK
Progression : 26/150 | OK
Progression : 27/150 | OK
Progression : 28/150 | OK
Progression : 29/150 | OK
Progression : 30/150 | OK
Progression : 31/150 | OK
Progression : 32/150 | OK
Progression : 33/150 | OK
Progression : 34/150 | OK
Progression : 35/150 | OK
Progression : 36/150 | OK
Progression : 37/150 | OK
Progress

,url,Titre,Ville,Pays,ID,Type,Date de publication,Texte de l'offre
0,https://careers.loreal.com/fr_FR/jobs/JobDetai...,Trade Marketing Trainee - L'Oréal Dermatologic...,"Espoo, Southwest Finland",Finland,198643,Stage,2025-02-12,Stage\n ...
1,https://careers.loreal.com/fr_FR/jobs/JobDetai...,Internship Marketing (Paid) - Per February 2026,"Hoofddorp, North Holland",Netherlands,217197,Stage,2025-09-03,Stage\n ...
2,https://careers.loreal.com/fr_FR/jobs/JobDetai...,Stage Ingénieur Assurance Qualité (H/F),"Libramont, Wallonia",Belgium,223156,Stage,2025-04-09,Stage\n ...
3,https://careers.loreal.com/fr_FR/jobs/JobDetai...,PRAKTIKUM (M/W/D) Human Resources - Start März...,"Vienna, Vienna",Austria,217313,Stage,2025-09-04,Stage\n ...
4,https://careers.loreal.com/fr_FR/jobs/JobDetai...,Digital Data Internship,"Prague, Praha",Czech Republic,227812,Stage,2025-12-12,Stage\n ...


In [39]:
# Nettoyage et filtrage des offres L'oreal
loreal_jobs_full.dropna(inplace=True)
# garder uniquement les offres de stage en France avec la colonne pays 
loreal_jobs_full = loreal_jobs_full[(loreal_jobs_full['Pays'].str.lower() == 'france')]
print(f"Offres après nettoyage : {len(loreal_jobs_full)}")
loreal_jobs_full.head()

Offres après nettoyage : 67


,url,Titre,Ville,Pays,ID,Type,Date de publication,Texte de l'offre
35,https://careers.loreal.com/fr_FR/jobs/JobDetai...,Stage de 3 mois minimum - À partir de Janvier ...,"Pau, Aquitaine-Limousin-Poitou-Charentes",France,223831,Stage,2025-10-30,Stage\n ...
36,https://careers.loreal.com/fr_FR/jobs/JobDetai...,Stage de 6 mois - Ingénieur(e) EHS Santé Sécur...,"Pau, Aquitaine-Limousin-Poitou-Charentes",France,226736,Stage,2025-12-01,Stage\n ...
70,https://careers.loreal.com/fr_FR/jobs/JobDetai...,STAGE 6 MOIS - À PARTIR DÉBUT MARS 2026 - Ingé...,"Aulnay-sous-Bois, Île-de-France",France,221392,Stage,2026-03-02,Stage\n ...
71,https://careers.loreal.com/fr_FR/jobs/JobDetai...,STAGE 6 MOIS - À partir de mars 2026 – Flux QH...,"Cambrai, Nord-Pas-de-Calais-Picardie",France,218042,Stage,2026-02-03,Stage\n ...
72,https://careers.loreal.com/fr_FR/jobs/JobDetai...,STAGE 6 MOIS - À partir de janvier 2026 – Perf...,"Cambrai, Nord-Pas-de-Calais-Picardie",France,218040,Stage,2026-01-26,Stage\n ...


In [42]:
# Nettoyer la colonne Ville pour garder uniquement la ville
loreal_jobs_stages = loreal_jobs_full.copy()
loreal_jobs_stages['Ville'] = loreal_jobs_stages['Ville'].str.split(',').str[0].str.strip()

print("Villes nettoyées !")
print("\nExemples L'oreal :")
print(loreal_jobs_stages[['Titre', 'Ville']].head())


Villes nettoyées !

Exemples L'oreal :
                                                Titre             Ville
35  Stage de 3 mois minimum - À partir de Janvier ...               Pau
36  Stage de 6 mois - Ingénieur(e) EHS Santé Sécur...               Pau
70  STAGE 6 MOIS - À PARTIR DÉBUT MARS 2026 - Ingé...  Aulnay-sous-Bois
71  STAGE 6 MOIS - À partir de mars 2026 – Flux QH...           Cambrai
72  STAGE 6 MOIS - À partir de janvier 2026 – Perf...           Cambrai


In [44]:
loreal_jobs_stages.head(10)

,url,Titre,Ville,Pays,ID,Type,Date de publication,Texte de l'offre
35,https://careers.loreal.com/fr_FR/jobs/JobDetai...,Stage de 3 mois minimum - À partir de Janvier ...,Pau,France,223831,Stage,2025-10-30,Stage\n ...
36,https://careers.loreal.com/fr_FR/jobs/JobDetai...,Stage de 6 mois - Ingénieur(e) EHS Santé Sécur...,Pau,France,226736,Stage,2025-12-01,Stage\n ...
70,https://careers.loreal.com/fr_FR/jobs/JobDetai...,STAGE 6 MOIS - À PARTIR DÉBUT MARS 2026 - Ingé...,Aulnay-sous-Bois,France,221392,Stage,2026-03-02,Stage\n ...
71,https://careers.loreal.com/fr_FR/jobs/JobDetai...,STAGE 6 MOIS - À partir de mars 2026 – Flux QH...,Cambrai,France,218042,Stage,2026-02-03,Stage\n ...
72,https://careers.loreal.com/fr_FR/jobs/JobDetai...,STAGE 6 MOIS - À partir de janvier 2026 – Perf...,Cambrai,France,218040,Stage,2026-01-26,Stage\n ...
73,https://careers.loreal.com/fr_FR/jobs/JobDetai...,STAGE 6 MOIS - À partir de avril 2026 – Perfor...,Cambrai,France,225232,Stage,2026-01-12,Stage\n ...
74,https://careers.loreal.com/fr_FR/jobs/JobDetai...,STAGE 6 MOIS - À partir de janvier 2026 – Qual...,Cambrai,France,218068,Stage,2026-01-05,Stage\n ...
90,https://careers.loreal.com/fr_FR/jobs/JobDetai...,STAGE DE 6 MOIS - A partir de janvier/février ...,Rambouillet,France,224557,Stage,2025-12-01,Stage\n ...
91,https://careers.loreal.com/fr_FR/jobs/JobDetai...,Stagiaire Amélioration Continue et Digitalisat...,Aulnay-sous-Bois,France,215574,Stage,2025-10-01,Stage\n ...
92,https://careers.loreal.com/fr_FR/jobs/JobDetai...,STAGE 6 MOIS - À partir de février 2026 – Supp...,FR - Roye,France,219418,Stage,2026-02-01,Stage\n ...


In [ ]:
# Enregistrer les offres de stage L'oreal en CSV
loreal_jobs_stages.to_csv('offres/loreal_jobs_stages.csv', index=False, encoding='utf-8')
print("Fichier 'loreal_jobs_stages.csv' créé avec succès!")

Fichier 'loreal_jobs_stages.csv' créé avec succès!


In [3]:
orange = pd.read_csv('offres/orange_jobs_stages.csv', encoding='utf-8')
loreal = pd.read_csv('offres/loreal_jobs_stages.csv', encoding='utf-8')

orange['entreprise'] = 'Orange'
loreal['entreprise'] = 'L\'Oréal'

orange.to_csv('offres/orange_jobs_stages.csv', index=False, encoding='utf-8')
loreal.to_csv('offres/loreal_jobs_stages.csv', index=False, encoding='utf-8')